In [1]:
%%capture
import os
import pprint
import janitor
import pandas as pd
from convokit import Corpus, download
from pathlib import Path
from datasets import load_dataset
from textwrap import dedent

hf_token = os.getenv('HF_TOKEN')

In [2]:
gap = Corpus(filename = download("gap-corpus"))

convo_df = gap.get_conversations_dataframe().clean_names().reset_index(drop = False).drop(columns=['vectors'])
speaker_df = gap.get_speakers_dataframe().clean_names().reset_index(drop = False).drop(columns=['vectors'])
utt_df = gap.get_utterances_dataframe().clean_names().reset_index(drop = False).drop(columns=['vectors'])

Dataset already exists at /Users/jasminec/.convokit/saved-corpora/gap-corpus


In [3]:
ut_group_map  = (
    convo_df[['id', 'meta_group_number', 'meta_meeting_size', 'meta_meeting_length_in_minutes']]
    .rename(columns={
        'id': 'conversation_id',
        'meta_group_number': 'group_id',
        'meta_meeting_size': 'num_participants',
        'meta_meeting_length_in_minutes': 'duration'
        })
)

# merge w/ utterances 
utt_df = utt_df.merge(ut_group_map, on='conversation_id', how='left')

In [4]:
# full transcript generation
transcripts = []
convo = None
unlabeled_convo = None
group_id = None

for i, r in utt_df.iterrows():
    if group_id == r['group_id']:
        convo += f"{r['speaker']}: {r['text']}\n"
        unlabeled_convo += f"{r['text']}\n"
    elif group_id != r['group_id'] and group_id is not None:
        transcripts.append(
            {
                'group_id': group_id,
                'convo': convo,
                'unlabeled_convo': unlabeled_convo,
            }
        )
        group_id = r['group_id']
        convo = f"{r['speaker']}: {r['text']}\n"
        unlabeled_convo = f"{r['text']}\n"
    else:
        group_id = r['group_id']
        convo = f"{r['speaker']}: {r['text']}\n"
        unlabeled_convo = f"{r['text']}\n"

if group_id is not None and convo is not None:
    transcripts.append(
        {
            'group_id': group_id,
            'convo': convo,
            'unlabeled_convo': unlabeled_convo,
        }
    )

In [ ]:
# excerpts for evals.
chunk_size = 40

meeting_size_map = (
    convo_df[["meta_group_number", "meta_meeting_size"]]
    .drop_duplicates()
    .assign(
        meta_group_number=lambda d: d["meta_group_number"].astype(str),
        meta_meeting_size=lambda d: pd.to_numeric(d["meta_meeting_size"], errors="coerce"),
    )
    .set_index("meta_group_number")["meta_meeting_size"]
    .to_dict()
)

excerpts = []
for t in transcripts:
    group_id = str(t["group_id"])

    labeled_lines = [line for line in t["convo"].splitlines() if line.strip()]
    unlabeled_lines = [
        line.split(":", 1)[1].strip() if ":" in line else line
        for line in labeled_lines
    ]

    for chunk_ix, start in enumerate(range(0, len(labeled_lines), chunk_size)):
        end = min(start + chunk_size, len(labeled_lines))
        labeled_chunk = labeled_lines[start:end]
        unlabeled_chunk = unlabeled_lines[start:end]

        speakers = sorted({line.split(":", 1)[0] for line in labeled_chunk if ":" in line})
        gt = meeting_size_map.get(group_id)
        global_speaker_count = int(gt) if pd.notna(gt) else len(speakers)

        excerpts.append(
            {
                "source": f"{group_id}:{start + 1}-{end}",
                "group_id": group_id,
                "chunk_ix": chunk_ix,
                "line_start": start + 1,
                "line_end": end,
                "num_lines": len(labeled_chunk),
                "speakers": set(speakers),
                "num_speakers": len(speakers),
                "global_speaker_count": global_speaker_count,
                "baseline_text": "\n".join(labeled_chunk) + "\n",
                "unlabeled_text": "\n".join(unlabeled_chunk) + "\n",
            }
        )

len(excerpts)

331

In [7]:
len(excerpts)

214

# Evals

In [124]:
import asyncio
import ast
import importlib
import json
import re
import time
import math
from pathlib import Path

import pandas as pd
import utils.utils as utils_module

importlib.reload(utils_module)
from utils.utils import send_openrouter_request_async

if "excerpts" not in globals() or excerpts is None:
    excerpts_path = Path("excerpts.csv")
    if not excerpts_path.exists():
        raise NameError(
            "`excerpts` is not defined and excerpts.csv was not found. "
            "Run the excerpt-generation cell first."
        )

    excerpts_df = pd.read_csv(excerpts_path)

    if "speakers" in excerpts_df.columns:
        def _parse_speakers(value):
            if pd.isna(value):
                return []
            if isinstance(value, (list, set, tuple)):
                return list(value)
            try:
                parsed = ast.literal_eval(str(value))
                if isinstance(parsed, set):
                    return sorted(parsed)
                if isinstance(parsed, (list, tuple)):
                    return list(parsed)
            except Exception:
                pass
            return [str(value)]

        excerpts_df["speakers"] = excerpts_df["speakers"].apply(_parse_speakers)

    excerpts = excerpts_df.to_dict("records")

n_eval_per_model = len(excerpts)
concurrency = 18
max_tokens = 100

model_ids = [
    # add large open source models - need to check if reasoning can be turned off
    # # 'moonshotai/kimi-k2.5', # 1T A32B (https://huggingface.co/moonshotai/Kimi-K2.5) # reasoning is mandatory
    # 'minimax/minimax-m2.7', # 230B A10B # mandatory reasoning 
    # 'z-ai/glm-5.1', # 754B A40B

    # # deepseek
    # # 'deepseek/deepseek-r1-0528', #671B A 37B # reasoning is mandatory
    'deepseek/deepseek-v3.2', # 685B

    # # qwen
    'qwen/qwen-2.5-7b-instruct',
    # 'qwen/qwen-2.5-14b-instruct',
    # 'qwen/qwen2.5-32b-instruct',
    'qwen/qwen-2.5-72b-instruct',
    # 'qwen/qwen3-30b-a3b-instruct-2507',
    'qwen/qwen3-235b-a22b-2507',

    # # qwen 3.5
    # 'qwen/qwen3.5-9b',
    # 'qwen/qwen3.5-35b-a3b',
    # 'qwen/qwen3.5-27b',
    # 'qwen/qwen3.5-122b-a10b',
    # 'qwen/qwen3.5-397b-a17b',

    # # mistral
    # # "mistralai/mixtral-8x7b-instruct",
    # # "mistralai/mixtral-8x22b-instruct",
    
    # # gemini # reasoning is fixed to be on 
    # # 'google/gemini-2.5-pro',
    # # 'google/gemini-3.1-pro-preview',

    # # anthropic
    'anthropic/claude-opus-4',
    'anthropic/claude-opus-4.7',


    # # gpt
    'openai/gpt-3.5-turbo',
    'openai/gpt-4',
    # 'openai/gpt-4o',
    # 'openai/gpt-5.2',
    'openai/gpt-5.4', # keep this as a pace-setter
    # 'openai/gpt-5-mini'
]

use_labeled_text = True

def parse_first_int(text):
    m = re.search(r"\b(\d+)\b", str(text))
    return int(m.group(1)) if m else None

async def eval_one(model_id, example, semaphore):
    text_key = "baseline_text" if use_labeled_text else "unlabeled_text"
    excerpt_text = example[text_key]
    prompt = dedent(f"""You will be asked a question about a conversation and MUST RESPOND with ONLY a SINGLE NUMBER.

    Note: there are also symbols used to represent non-speech information:
    • “$” means laughter,
    • “%” represents coughing,
    • “#” indicates another noise.

    Here is the excerpt:
    ```
    {excerpt_text}
    ```
    How many distinct speakers were in this conversation?
    Answer:
    """)

    messages = [{"role": "user", "content": prompt}]

    async with semaphore:
        t0 = time.perf_counter()
        try:
            response, reason, refusal, provider = await send_openrouter_request_async(
                messages,
                model=model_id,
                temperature = 0,
                max_tokens=max_tokens,
                reasoning={"effort": "none"},
            )
            error = None
        except Exception as exc:
            response, reason, refusal, provider = "", None, None, None
            error = str(exc)
        # elapsed_s = time.perf_counter() - t0

    pred = parse_first_int(response or "")
    return {
        "model_id": model_id,
        "task_variant": "with_labels" if use_labeled_text else "without_labels",
        "source": example["source"],
        "group_id": example["group_id"],
        "chunk_ix": example["chunk_ix"],
        "line_start": example["line_start"],
        "line_end": example["line_end"],
        "num_lines": example["num_lines"],
        "predicted_speakers": pred,
        "chunk_num_speakers": example["num_speakers"],
        "is_correct": None if pred is None else pred == example["num_speakers"],
        "chunk_speakers": sorted(example["speakers"]),
        "excerpt_text": excerpt_text,
        "response": response,
        "reasoning": reason,
        "refusal": refusal,
        "provider": provider,
        "error": error,
    }

async def run_eval(examples, models, max_examples, max_concurrency):
    sem = asyncio.Semaphore(max_concurrency)
    tasks = [
        eval_one(model_id, e, sem)
        for model_id in models
        for e in examples[:max_examples]
    ]
    return await asyncio.gather(*tasks)

results = await run_eval(excerpts, model_ids, n_eval_per_model, concurrency)
results_df = pd.DataFrame(results)
results_df["is_incomplete"] = results_df["predicted_speakers"].isna()
results_df["is_correct"] = results_df["is_correct"].astype("boolean")

# omit cases where all models except one are wrong (completed answers only)
n_models = results_df["model_id"].nunique()
min_wrong_models = n_models - 1

omit_source_stats = (
    results_df[~results_df["is_incomplete"]]
    .groupby("source", as_index=False)
    .agg(
        n_complete=("model_id", "nunique"),
        n_wrong=("is_correct", lambda s: (s == False).sum()),
    )
)

omit_sources = omit_source_stats.loc[
    (omit_source_stats["n_complete"] == n_models)
    & (omit_source_stats["n_wrong"] >= min_wrong_models),
    "source",
]

results_df_filtered = results_df[~results_df["source"].isin(omit_sources)].copy()
results_df_scored = results_df_filtered[~results_df_filtered["is_incomplete"]].copy()

print(
    f"Dropped {len(omit_sources)} excerpts (>= {min_wrong_models}/{n_models} wrong models, full completion)."
)
print(
    f"Excluded {results_df_filtered['is_incomplete'].sum()} incomplete answers from accuracy calculations."
)

summary_base = (
    results_df_filtered.groupby("model_id", as_index=False)
    .agg(
        task_variant=("task_variant", "first"),
        n_total=("source", "count"),
        n_incomplete=("is_incomplete", "sum"),
        n_api_errors=("error", lambda s: s.notna().sum()),
    )
)

summary_acc = (
    results_df_scored.groupby("model_id", as_index=False)
    .agg(accuracy=("is_correct", "mean"), n_scored=("source", "count"))
)

summary_df = (
    summary_base
    .merge(summary_acc, on="model_id", how="left")
    .sort_values(["accuracy", "n_api_errors"], ascending=[False, True])
)


# check
# pd.set_option("display.max_colwidth", None)
# pd.set_option("display.max_columns", None)
display(summary_df)
display(
    results_df_filtered[
        [
            "model_id",
            "provider",
            "task_variant",
            "is_incomplete",
            "is_correct",
            "source",
            "group_id",
            "chunk_ix",
            "line_start",
            "line_end",
            "num_lines",
            "chunk_num_speakers",
            "chunk_speakers",
            "excerpt_text",
            "predicted_speakers",
            "response",
            "reasoning",
            "refusal",
            "error",
        ]
    ].head(10)
)

Dropped 0 excerpts (>= 8/9 wrong models, full completion).
Excluded 14 incomplete answers from accuracy calculations.


,model_id,task_variant,n_total,n_incomplete,n_api_errors,accuracy,n_scored
1,anthropic/claude-opus-4.7,with_labels,214,0,0,1.0,214
6,qwen/qwen-2.5-72b-instruct,with_labels,214,0,0,1.0,214
0,anthropic/claude-opus-4,with_labels,214,0,0,0.995327,214
5,openai/gpt-5.4,with_labels,214,1,0,0.995305,213
4,openai/gpt-4,with_labels,214,0,0,0.990654,214
8,qwen/qwen3-235b-a22b-2507,with_labels,214,13,0,0.960199,201
2,deepseek/deepseek-v3.2,with_labels,214,0,0,0.953271,214
7,qwen/qwen-2.5-7b-instruct,with_labels,214,0,0,0.434579,214
3,openai/gpt-3.5-turbo,with_labels,214,0,0,0.28972,214


,model_id,provider,task_variant,is_incomplete,is_correct,source,group_id,chunk_ix,line_start,line_end,num_lines,chunk_num_speakers,chunk_speakers,excerpt_text,predicted_speakers,response,reasoning,refusal,error
0,deepseek/deepseek-v3.2,SiliconFlow,with_labels,False,True,1:1-40,1,0,1,40,40,3,"[1.Blue, 1.Green, 1.Pink]","1.Pink: ""So what did everyone do as one""\n1.Bl...",3.0,3,None,None,None
1,deepseek/deepseek-v3.2,SiliconFlow,with_labels,False,True,1:41-80,1,1,41,80,40,3,"[1.Blue, 1.Green, 1.Pink]","1.Pink: ""Okay no I really dont it doesnt matte...",3.0,3,None,None,None
2,deepseek/deepseek-v3.2,SiliconFlow,with_labels,False,False,1:81-120,1,2,81,120,40,3,"[1.Blue, 1.Green, 1.Pink]","1.Blue: ""And then my five was a knife""\n1.Blue...",4.0,4,None,None,None
3,deepseek/deepseek-v3.2,SiliconFlow,with_labels,False,True,1:121-160,1,3,121,160,40,3,"[1.Blue, 1.Green, 1.Pink]","1.Pink: ""Hm""\n1.Green: ""I did compression kit""...",3.0,3,None,None,None
4,deepseek/deepseek-v3.2,SiliconFlow,with_labels,False,True,1:161-200,1,4,161,200,40,3,"[1.Blue, 1.Green, 1.Pink]","1.Pink: ""$""\n1.Blue: ""Yeah I I literally I put...",3.0,3,None,None,None
5,deepseek/deepseek-v3.2,SiliconFlow,with_labels,False,True,1:201-240,1,5,201,240,40,3,"[1.Blue, 1.Green, 1.Pink]","1.Green: ""We have a knife""\n1.Blue: ""Oh yeah""\...",3.0,3,None,None,None
6,deepseek/deepseek-v3.2,SiliconFlow,with_labels,False,True,1:241-280,1,6,241,280,40,3,"[1.Blue, 1.Green, 1.Pink]","1.Pink: ""Knife""\n1.Pink: ""Um""\n1.Blue: ""My ten...",3.0,3,None,None,None
7,deepseek/deepseek-v3.2,SiliconFlow,with_labels,False,True,1:281-320,1,7,281,320,40,3,"[1.Blue, 1.Green, 1.Pink]","1.Blue: ""Okay so do we want to put um""\n1.Pink...",3.0,3,None,None,None
8,deepseek/deepseek-v3.2,SiliconFlow,with_labels,False,True,1:321-349,1,8,321,349,29,3,"[1.Blue, 1.Green, 1.Pink]","1.Blue: ""Do you want to put the chocolate bar ...",3.0,3,None,None,None
9,deepseek/deepseek-v3.2,SiliconFlow,with_labels,False,True,10:1-40,10,0,1,40,40,2,"[10.Orange, 10.Pink]","10.Orange: ""All right""\n10.Pink: ""Uh I found t...",2.0,2,None,None,None


In [125]:
# pprint.pprint(results_df.head(1)['reasoning'][0])
results_df.to_csv('labeled_eval_gap.csv', index = False)
# pprint.pprint(results_df.head(1)['excerpt_text'][0])

In [104]:
incomplete = []
for i, r in results_df.iterrows():
    if r['model_id'] == 'deepseek/deepseek-v3.2':
        print(r)
        incomplete.append({'is_incomplete': r['is_incomplete'],'provider': r['provider'], 'model_id': r['model_id'], 'reasoning': r['reasoning']})

In [81]:
for i in incomplete:
    if i['model_id'] == 'deepseek/deepseek-v3.2':
        print(i['is_incomplete'], i['provider'], i['reasoning'])

False SiliconFlow We are given a conversation excerpt and need to determine the number of speakers actively and directly participating. We must output only a single number.

First, we need to analyze the conversation. The excerpt contains lines of dialogue, some with laughter symbols "$", coughing "%", and noise "#". But note: the example shows that non-speech symbols like "$" are not separate speakers; they are just sounds made by speakers. So we need to identify distinct speakers based on the dialogue.

We don't have speaker labels; we only have lines of quoted text. Each line could be from a different speaker or the same speaker. We need to infer the number of distinct speakers from the pattern of conversation.

In such tasks, typically we look for cues like changes in voice, content, and turn-taking. But without explicit labels, we might assume that each new line could be a new speaker, but that's not necessarily true. Sometimes the same speaker might have consecutive lines. Howeve

In [55]:
n_models = results_df["model_id"].nunique()

# Drop excerpts only when every model completed and all were wrong.
by_source = (
    results_df.groupby("source", as_index=False)
    .agg(
        n_models_seen=("model_id", "nunique"),
        n_completed=("predicted_speakers", lambda s: s.notna().sum()),
        n_correct=("is_correct", lambda s: (s == True).sum()),
    )
)

all_wrong_sources = by_source.loc[
    (by_source["n_models_seen"] == n_models)
    & (by_source["n_completed"] == n_models)
    & (by_source["n_correct"] == 0),
    "source"
]

# remove
results_df_filtered = results_df[~results_df["source"].isin(all_wrong_sources)].copy()

print(f"Total excerpts: {results_df['source'].nunique()}")
print(f"Dropped fully-complete all-wrong excerpts: {len(all_wrong_sources)}")
print(f"Remaining excerpts: {results_df_filtered['source'].nunique()}")

summary_filtered = (
    results_df_filtered.groupby("model_id", as_index=False)
    .agg(
        task_variant=("task_variant", "first"),
        n=("source", "count"),
        accuracy=("is_correct", "mean"),
        n_api_errors=("error", lambda s: s.notna().sum()),
    )
    .sort_values(["accuracy", "n_api_errors"], ascending=[False, True])
)

display(summary_filtered)

# droped excerpts
dropped_excerpt_details = (
    results_df[results_df["source"].isin(all_wrong_sources)]
    .drop_duplicates("source")[
        ["source", "group_id", "chunk_ix", "line_start", "line_end", "chunk_num_speakers", "excerpt_text"]
    ]
    .sort_values(["group_id", "chunk_ix"])
)
display(dropped_excerpt_details)

# model level preds on dropped excerpts
dropped_predictions = (
    results_df[results_df["source"].isin(all_wrong_sources)]
    .sort_values(["source", "model_id"])[
        ["source", "model_id", "predicted_speakers", "chunk_num_speakers", "is_correct", "response"]
    ]
) 
display(dropped_predictions) 

Total excerpts: 20
Dropped fully-complete all-wrong excerpts: 0
Remaining excerpts: 20


,model_id,task_variant,n,accuracy,n_api_errors
2,openai/gpt-5.4-pro,without_labels,20,0.666667,0
1,mistralai/mixtral-8x7b-instruct,without_labels,20,0.55,0
4,openai/gpt-oss-20b,without_labels,20,0.5,0
7,qwen/qwen3-235b-a22b-2507,without_labels,20,0.5,0
3,openai/gpt-oss-120b,without_labels,20,0.466667,0
5,qwen/qwen-2.5-72b-instruct,without_labels,20,0.45,0
8,qwen/qwen3-30b-a3b-instruct-2507,without_labels,20,0.25,0
0,mistralai/mixtral-8x22b-instruct,without_labels,20,0.15,0
6,qwen/qwen-2.5-7b-instruct,without_labels,20,0.1,0


,source,group_id,chunk_ix,line_start,line_end,chunk_num_speakers,excerpt_text


,source,model_id,predicted_speakers,chunk_num_speakers,is_correct,response


In [ ]:
# export
results_df.to_csv('full_eval_gap.csv', index=False, na_rep='N/A')

In [ ]:
# notif
from utils.utils import send_slack

variant = summary_filtered["task_variant"].iloc[0] if len(summary_filtered) else "unknown"
filtered = results_df_filtered.copy()

per_model = (
    filtered.assign(
        completed=lambda d: d["predicted_speakers"].notna(),
        correct=lambda d: d["is_correct"].fillna(False) & d["predicted_speakers"].notna(),
    )
    .groupby("model_id", as_index=False)
    .agg(
        n_total=("source", "count"),
        n_completed=("completed", "sum"),
        n_correct=("correct", "sum"),
    )
)
per_model["success_rate"] = per_model["n_correct"] / per_model["n_completed"].where(per_model["n_completed"] > 0)

model_order = {m: i for i, m in enumerate(summary_filtered["model_id"].tolist())}
per_model["_order"] = per_model["model_id"].map(model_order).fillna(10**9)
per_model = per_model.sort_values("_order")

n_no_response = int((~filtered["predicted_speakers"].notna()).sum())

summary_lines = [
    "dataset: GAP",
    f"mode: speaker_count ({variant})",
    f"models: {summary_filtered['model_id'].nunique()}",
    f"excerpts: {filtered['source'].nunique()} (raw {results_df['source'].nunique()}, dropped {len(all_wrong_sources)})",
    f"no response: {n_no_response}/{len(filtered)}",
]

model_lines = [
    f"- {r.model_id}: {'*n/a*' if pd.isna(r.success_rate) else f'*{r.success_rate:.1%}*'} ({int(r.n_correct)}/{int(r.n_completed)})"
    for r in per_model.itertuples()
]

slack_text = "\n".join([*summary_lines, "", "model breakdown:", *model_lines])
resp = send_slack(slack_text)
print(f"Slack status: {resp.status_code}")
print(resp.text)
slack_text

Slack status: 200
ok


'dataset: GAP\nmode: speaker_count (without_labels)\nmodels: 9\nexcerpts: 20 (raw 20, dropped 0)\nno response: 28/180\n\nmodel breakdown:\n- openai/gpt-5.4-pro: *66.7%* (6/9)\n- mistralai/mixtral-8x7b-instruct: *55.0%* (11/20)\n- openai/gpt-oss-20b: *50.0%* (4/8)\n- qwen/qwen3-235b-a22b-2507: *50.0%* (10/20)\n- openai/gpt-oss-120b: *46.7%* (7/15)\n- qwen/qwen-2.5-72b-instruct: *45.0%* (9/20)\n- qwen/qwen3-30b-a3b-instruct-2507: *25.0%* (5/20)\n- mistralai/mixtral-8x22b-instruct: *15.0%* (3/20)\n- qwen/qwen-2.5-7b-instruct: *10.0%* (2/20)'